## Step 1: Clean Up & Prepare the Data
Before feeding data into a pipeline, we need to handle the string artifacts (like `'NaN?'` or `'Null'`) and convert them into true Python `NaN` values

In [24]:
# Import the dataset
import pandas as pd
import numpy as np
from narwhals import median
from pyexpat.errors import XML_ERROR_BINARY_ENTITY_REF
from sklearn.externals.array_api_extra import one_hot
# Machine Learning libraries
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import joblib

In [3]:
df = pd.read_csv('data/retail_sales.csv')
df.head()

,Date,Category,Sales,Quantity,Profit,Region
0,1/1/2023,Electronics,1149.014246,11.0,383.664245,North
1,1/1/2023,Clothing,958.520710,7.0,224.054049,East
2,1/1/2023,Home Goods,1473.763845,2.0,466.593090,South
3,1/1/2023,Sports,1230.230419,6.0,123.310460,West
4,1/1/2023,NaN?,828.585950,12.0,88.591355,East


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1825 entries, 0 to 1824
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Date      1825 non-null   str    
 1   Category  1821 non-null   str    
 2   Sales     1823 non-null   float64
 3   Quantity  1820 non-null   float64
 4   Profit    1825 non-null   float64
 5   Region    1820 non-null   str    
dtypes: float64(3), str(3)
memory usage: 124.3 KB


In [5]:
# Standardize messy string values into true missing values (NaN)
messy_null_values = ['NAN?','Null','Nan']
df['Category'] =  df['Category'].replace(messy_null_values , np.nan)
df['Region'] = df['Region'].replace(messy_null_values , np.nan)


In [9]:
# Feature Engineering: Extract date features if relevent
df['Date'] = pd.to_datetime(df['Date'])
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.day_of_week
df.head()

,Date,Category,Sales,Quantity,Profit,Region,Month,DayOfWeek
0,2023-01-01,Electronics,1149.014246,11.0,383.664245,North,1,6
1,2023-01-01,Clothing,958.520710,7.0,224.054049,East,1,6
2,2023-01-01,Home Goods,1473.763845,2.0,466.593090,South,1,6
3,2023-01-01,Sports,1230.230419,6.0,123.310460,West,1,6
4,2023-01-01,NaN?,828.585950,12.0,88.591355,East,1,6


In [8]:
# Features and target variables
X = df.drop(columns=['Profit' , 'Date'])
y = df['Profit']


## Step 2: Build the Machine Learning Pipeline
Pipeline separates numeric features from categorical features, applies the correct math to each, and feeds them into an estimator **(like a Random Forest or Gradient Boosting Regressor)**.

In [27]:
# 1. Split data into training and testing sets
X_train , X_test , y_train , y_test =  train_test_split(X , y , test_size=0.20 , random_state=42)

# 2. Group columns by data type
numerical_features = ['Sales' , 'Quantity' , 'Month' , 'DayOfWeek']
catgorical_features = ['Category' , 'Region']

# 3. Create Preprocessing steps for Numeric Data (Impute missing -> Scale numbers)
numeric_transformer = Pipeline(steps=[
    ('imputer' , SimpleImputer(strategy='median')),
    ('scaler' , StandardScaler())
])

# 4. Create Preprocessing steps for Categorical Data (Impute missing -> One-Hot Encode)
catgorical_transformer = Pipeline(steps=[
    ('imputer' , SimpleImputer(strategy='most_frequent')),
    ('onehot' , OneHotEncoder(handle_unknown='ignore'))
])

# 5. Combine transformers into a single preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num' , numeric_transformer , numerical_features),
        ('cat' , catgorical_transformer , catgorical_features)
    ]
    )

# 6. Assemble the final unified pipeline (Preprocessor + Model)
pipeline = Pipeline(steps=[
    ('preprocessor' , preprocessor),
    ('regressor' , RandomForestRegressor(n_estimators=100 , random_state=42))
])

## Step 3: Train, Evaluate, and Save

In [28]:
# Train the entire pipeline.
pipeline.fit(X_train , y_train)

# Evaluate performance on test data
y_pred = pipeline.predict(X_test)
print("\n--- Pipeline evaluation Metrics ---")
print(f"Maen Abosolute Error (MAE) : {mean_absolute_error(y_test, y_pred):.2f}")
print(f"R-squared (R²) Score: {r2_score(y_test, y_pred):.2f}")

# Save the unified pipeline for deployment
joblib.dump(pipeline , 'model/retail_profit_pipeline.pkl')
print("Model saved successfully!")


--- Pipeline evaluation Metrics ---
Maen Abosolute Error (MAE) : 78.84
R-squared (R²) Score: 0.29


['model/retail_profit_pipeline.pkl']

In [30]:
# Simulating a live prediction request coming from a web dashboard
new_transaction = pd.DataFrame([{
    'Category': 'Electronics',
    'Sales': 1200.50,
    'Quantity': 5.0,
    'Region': 'North',
    'Month': 7,
    'DayOfWeek': 4
}])

# Load and predict inline
deployed_model = joblib.load('model/retail_profit_pipeline.pkl')
predicted_profit = deployed_model.predict(new_transaction)[0]
print(f"Predicted Transaction Profit: ${predicted_profit:.2f}")

Predicted Transaction Profit: $314.03
